In [ ]:
!git clone https://github.com/beshoyhakeem/Euro-Laws-Agent-Helper.git

Cloning into 'Euro-Laws-Agent-Helper'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 148 (delta 62), reused 133 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 440.68 KiB | 2.42 MiB/s, done.
Resolving deltas: 100% (62/62), done.


In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"puskas78","key":"831a4bb39cb72b84ff11cb104fd05fdb"}'}

In [ ]:
!mkdir -p ~/.kaggle

In [ ]:
!mv kaggle.json ~/.kaggle/

In [ ]:
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download puskas78/euro-laws-embed-chunks

Dataset URL: https://www.kaggle.com/datasets/puskas78/euro-laws-embed-chunks
License(s): CC0-1.0
 99% 2.52G/2.55G [00:50<00:00, 53.4MB/s]
100% 2.55G/2.55G [00:51<00:00, 53.6MB/s]


In [ ]:
!unzip euro-laws-embed-chunks -d dataset/

Archive:  euro-laws-embed-chunks.zip
  inflating: dataset/full_doc_embedd/Eurolex_Laws_Chunked_embedded/embeddings.npy  
  inflating: dataset/full_doc_embedd/Eurolex_Laws_Chunked_embedded/metadata_chunks.pkl  
  inflating: dataset/full_doc_embedd/Eurolex_Laws_Chunked_embedded/metadata_chunks_with_full_doc.pkl  
  inflating: dataset/full_doc_embedd/Eurolex_Laws_Chunked_embedded/text_chunks.pkl  
  inflating: dataset/minimal-metadata.json  


In [ ]:
!kaggle datasets download puskas78/eurlex-laws-cleaned-dataset

Dataset URL: https://www.kaggle.com/datasets/puskas78/eurlex-laws-cleaned-dataset
License(s): other
 95% 334M/350M [00:06<00:00, 36.3MB/s]
100% 350M/350M [00:06<00:00, 53.9MB/s]


In [ ]:
!unzip eurlex-laws-cleaned-dataset -d dataset/

Archive:  eurlex-laws-cleaned-dataset.zip
  inflating: dataset/Eurolex_Laws_Cleaned_Dataset.csv  


In [ ]:
!pip install weaviate-client[agents]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.7/618.7 kB 12.4 MB/s eta 0:00:00


In [ ]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType
import pickle , json
import numpy as np
import os
from tqdm import tqdm
import pandas as pd

In [ ]:
weaviate_url = "https://d7w2dbddszamdttojx0uiw.c0.europe-west3.gcp.weaviate.cloud"
weaviate_api_key = "RUdiVjY4R0FPdmZsVWRtcV9TUExlY2ZybUNXL2hieGlNRTJSSGNKZ3NUOVFpVkQrbFM5TEVablhRZUQwPV92MjAw"

In [ ]:
with open('/content/dataset/text_chunks.pkl','rb') as c:
    text_chunks = pickle.load(c)

In [ ]:
text_chunks[1]

'from the beginning of the financial year 2019, HAVE ADOPTED THIS DECISION: Article 1 1. For the general budget of the Union for the financial year 2019, the Flexibility Instrument shall be mobilised to provide the amount of EUR 178 715 475 in commitment appropriations in heading 1a (Competitiveness for growth and jobs) and EUR 985 629 138 in commitment appropriations in heading 3 (Security and Citizenship). The amounts referred to in the first subparagraph shall be used to reinforce key programmes for the competitiveness of the EU and finance measures to address the ongoing challenges of migration, refugee inflows and security threats. 2. On the basis of the expected payment profile, the payment appropriations corresponding to the mobilisation of the Flexibility Instrument shall be estimated as follows: (a) EUR 548 740 834 in 2019; (b) EUR 257 223 207 in 2020; (c) EUR 135 194 558 in 2021; (d) EUR 140 942 662 in 2022; (e) EUR 82 243 352 in 2023. The specific amounts of payment appropri

In [ ]:
with open('/content/dataset/embeddings.npy','rb') as e:
   emmbedings  = np.load(e)

In [ ]:
emmbedings[4]

array([-8.53590202e-03, -3.76913473e-02, -1.22885534e-03, -9.15960129e-03,
        1.97996944e-02,  2.51504667e-02,  3.74376178e-02,  6.85439855e-02,
        6.36209967e-03,  1.74044445e-02,  4.08854596e-02,  9.02884379e-02,
       -2.96431547e-03,  2.73880307e-02,  1.82241462e-02,  2.85172090e-02,
        2.06325436e-03,  3.27116773e-02, -5.86415641e-02, -2.41800733e-02,
       -5.17128920e-03,  4.44943905e-02, -1.53398905e-02,  1.63928606e-03,
        2.54880171e-02,  2.41088439e-02,  4.15303670e-02, -1.45152155e-02,
        2.60560773e-02,  6.34555891e-02,  2.09351853e-02, -1.98820364e-02,
       -1.33731104e-02, -5.28397737e-03,  2.13279282e-06, -2.37659290e-02,
       -2.39428598e-03, -7.68245012e-03, -9.14116390e-03,  4.49052360e-03,
       -4.95670475e-02, -7.30321109e-02, -4.69788201e-02, -1.32517575e-03,
       -8.80494528e-03, -4.04762663e-02, -1.57522578e-02, -1.93716828e-02,
        1.36236669e-02,  3.64997350e-02, -1.20038334e-02, -5.64790424e-03,
       -1.72740873e-02, -

In [ ]:
with open('/content/dataset/metadata_chunks.pkl','rb') as m:
    metadata_chunks = pickle.load(m)

In [ ]:
metadata_chunks[9881]

{'celex': '32018D0778',
 'act_name': 'Council Decision (CFSP) 2018/778 of 28 May 2018 amending Decision 2013/255/CFSP concerning restrictive measures against Syria',
 'act_type': 'Decision',
 'eurovoc': 'blockade; natural person; Syria; economic sanctions; legal person',
 'subject_matter': 'international affairs;  civil law;  Asia and Oceania',
 'legal_basis': '12016M029',
 'date': '2018-05-28',
 'authors': 'European Council',
 'status': 'In Force',
 'cites': '32017D917',
 'treaty': 'http://publications.europa.eu/resource/authority/treaty/TEU_2012',
 'additional_info': 'No additional information',
 'chunk_number': 6,
 'total_chunks': 7,
 'document_length': 16439,
 'full_doc': "29.5.2018 EN Official Journal of the European Union L 131/16 COUNCIL DECISION (CFSP) 2018/778 of 28 May 2018 amending Decision 2013/255/CFSP concerning restrictive measures against Syria THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on European Union, and in particular Article 29 thereof, Having 

In [ ]:
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate_api_key,
)

In [ ]:
weaviate_client.is_connected()

True

In [ ]:
weaviate_client.collections.delete("Euro_Laws")

In [ ]:
eur_laws = weaviate_client.collections.create(
    name="Euro_Laws",
    vector_config=Configure.Vectors.self_provided(),
    properties=[
        Property(name="text", data_type=DataType.TEXT),
        Property(name="celex", data_type=DataType.TEXT),
        Property(name="act_name", data_type=DataType.TEXT),
        Property(name="act_type", data_type=DataType.TEXT),
        Property(name="eurovoc", data_type=DataType.TEXT),
        Property(name="subject_matter", data_type=DataType.TEXT),
        Property(name="legal_basis", data_type=DataType.TEXT),
        Property(name="authors", data_type=DataType.TEXT),
        Property(name="status", data_type=DataType.TEXT),
        Property(name="cites", data_type=DataType.TEXT),
        Property(name="treaty", data_type=DataType.TEXT),
        Property(name="additional_info", data_type=DataType.TEXT),
        Property(name="chunk_number", data_type=DataType.INT),
        Property(name="total_chunks", data_type=DataType.INT),
        Property(name="document_length", data_type=DataType.INT),
    ],
)

# hybrid schema

In [ ]:
eur_laws = weaviate_client.collections.create(
    name="Euro_Laws_hybrid",
    vector_config=Configure.Vectors.self_provided(),
    inverted_index_config=Configure.InvertedIndex(
        bm25=Configure.BM25(
            k1=1.2,
            b=0.68
        )
    ),
    properties=[
        Property(
            name="text",
            data_type=DataType.TEXT,
            index_searchable=True,   # 🔥 VERY IMPORTANT
            index_filterable=False
        ),
        Property(name="celex", data_type=DataType.TEXT),
        Property(name="act_name", data_type=DataType.TEXT),
        Property(name="act_type", data_type=DataType.TEXT),
        Property(name="eurovoc", data_type=DataType.TEXT),
        Property(name="subject_matter", data_type=DataType.TEXT),
        Property(name="legal_basis", data_type=DataType.TEXT),
        Property(name="authors", data_type=DataType.TEXT),
        Property(name="status", data_type=DataType.TEXT),
        Property(name="cites", data_type=DataType.TEXT),
        Property(name="treaty", data_type=DataType.TEXT),
        Property(name="additional_info", data_type=DataType.TEXT),
        Property(name="chunk_number", data_type=DataType.INT),
        Property(name="total_chunks", data_type=DataType.INT),
        Property(name="document_length", data_type=DataType.INT),
    ],
)

AttributeError: type object 'Configure' has no attribute 'InvertedIndex'

In [ ]:
weaviate_client.collections.delete("Euro_Laws_hybrid")

In [ ]:
eur_laws = weaviate_client.collections.create(
    name="Euro_Laws_hybrid",
    vector_config=Configure.Vectors.self_provided(),
    inverted_index_config=Configure.inverted_index(   # ✅ lowercase method
        bm25_b=0.65,
        bm25_k1=1.2,
    ),
    properties=[
        Property(
            name="text",
            data_type=DataType.TEXT,
            index_searchable=True,
        ),
        Property(name="celex", data_type=DataType.TEXT),
        Property(name="act_name", data_type=DataType.TEXT),
        Property(name="act_type", data_type=DataType.TEXT),
        Property(name="eurovoc", data_type=DataType.TEXT),
        Property(name="subject_matter", data_type=DataType.TEXT),
        Property(name="legal_basis", data_type=DataType.TEXT),
        Property(name="authors", data_type=DataType.TEXT),
        Property(name="status", data_type=DataType.TEXT),
        Property(name="cites", data_type=DataType.TEXT),
        Property(name="treaty", data_type=DataType.TEXT),
        Property(name="additional_info", data_type=DataType.TEXT),
        Property(name="chunk_number", data_type=DataType.INT),
        Property(name="total_chunks", data_type=DataType.INT),
        Property(name="document_length", data_type=DataType.INT),
    ],
)

In [ ]:
# Batch insert
eur_laws = weaviate_client.collections.get("Euro_Laws_hybrid")

with eur_laws.batch.fixed_size(batch_size=64) as batch:
    for i in tqdm(range(len(text_chunks))):
        text = text_chunks[i]
        meta = metadata_chunks[i]
        vector = emmbedings[i].tolist()

        properties = {
            "text": text,
            "celex": str(meta.get("celex", "")),
            "act_name": str(meta.get("act_name", "")),
            "act_type": str(meta.get("act_type", "")),
            "eurovoc": str(meta.get("eurovoc", "")),
            "subject_matter": str(meta.get("subject_matter", "")),
            "legal_basis": str(meta.get("legal_basis", "")),
            "authors": str(meta.get("authors", "")),
            "status": str(meta.get("status", "")),
            "cites": str(meta.get("cites", "")),
            "treaty": str(meta.get("treaty", "")),
            "additional_info": str(meta.get("additional_info", "")),
            "chunk_number": int(meta.get("chunk_number", 0)),
            "total_chunks": int(meta.get("total_chunks", 0)),
            "document_length": int(meta.get("document_length", 0)),
        }

        batch.add_object(
            properties=properties,
            vector=vector
        )

 35%|███▌      | 244736/694515 [27:41<48:38, 154.13it/s]INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "upstream connect error or disconnect/reset before headers. retried and the latest reset reason: remote connection failure, transport failure reason: delayed connect error: Connection refused"
	debug_error_string = "UNKNOWN:Error received from peer ipv4:34.111.245.21:443 {grpc_message:"upstream connect error or disconnect/reset before headers. retried and the latest reset reason: remote connection failure, transport failure reason: delayed connect error: Connection refused", grpc_status:14}"
>. Retrying with exponential backoff in 1 seconds
INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "upstream connect error or disconnect/reset before headers. retried and the latest reset reason: remote

# uploading collection 2


In [ ]:
law = pd.read_csv('/content/dataset/Eurolex_Laws_Cleaned_Dataset.csv')

In [ ]:
law.iloc[84186]

,84186
CELEX,31984R1291
Act_name,Commission Regulation (EEC) No 1291/84 of 10 M...
Act_type,Regulation
Status,Status not specified
EUROVOC,EUROVOC not specified
Subject_matter,Subject matter not specified
Treaty,Treaty not specified
Legal_basis_celex,Legal basis not specified
Authors,European Commission
Procedure_number,No procedure number


In [ ]:
eur_docs = weaviate_client.collections.create(
    name="Euro_Law_Documents",
    vector_config=None,  # no vector needed
    properties=[
        Property(name="celex", data_type=DataType.TEXT, index_filterable=True),
        Property(name="full_doc", data_type=DataType.TEXT),
        Property(name="status", data_type=DataType.TEXT),
        Property(name="act_type", data_type=DataType.TEXT),
        Property(name="treaty", data_type=DataType.TEXT),
        Property(name="text_length", data_type=DataType.INT)
    ],
)

In [ ]:
eur_docs = weaviate_client.collections.get("Euro_Law_Documents")

with eur_docs.batch.fixed_size(batch_size=100) as batch:

    for _, row in tqdm(law.iterrows(), total=len(law)):

        batch.add_object(
            properties={
                "celex": str(row["CELEX"]),
                "full_doc": str(row["act_raw_text"]),
                "status": str(row["Status"]),
                "act_type": str(row["Act_type"]),
                "treaty": str(row["Treaty"]),
                "text_length": int(row["text_length"]),
            }
        )

 34%|███▍      | 46100/134608 [09:55<20:38, 71.46it/s]INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "upstream connect error or disconnect/reset before headers. retried and the latest reset reason: remote connection failure, transport failure reason: delayed connect error: Connection refused"
	debug_error_string = "UNKNOWN:Error received from peer ipv4:34.111.245.21:443 {grpc_status:14, grpc_message:"upstream connect error or disconnect/reset before headers. retried and the latest reset reason: remote connection failure, transport failure reason: delayed connect error: Connection refused"}"
>. Retrying with exponential backoff in 1 seconds
INFO:weaviate-client:Batch objects received exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "upstream connect error or disconnect/reset before headers. retried and the latest reset reason: remote c

In [ ]:
eur_docs.aggregate.over_all(total_count=True)

AggregateReturn(properties={}, total_count=134608)

# to retrive with celex

In [ ]:
from weaviate.classes.query import Filter

eur_docs = weaviate_client.collections.get("Euro_Law_Documents")

response = eur_docs.query.fetch_objects(
    filters=Filter.by_property("celex").equal("31984R1291"),
    limit=1
)

r = response.objects[0].properties

In [ ]:
r['celex']

'31984R1291'

In [ ]:
print(weaviate.__version__)

4.20.1
